In [1]:
import os
import re
import math
import numpy as np
import pandas as pd

from collections import defaultdict
from statsmodels.stats.multitest import multipletests

# ============================================================
# CONFIG
# ============================================================

INPUT_DIR = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_DEG_results"
OUTPUT_DIR = os.path.join(INPUT_DIR, "Processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# All cell types are chunked for SEA-AD (no single-file Mic like 427)
CHUNKED_CELLTYPES = ["Ast", "Mic", "Oli", "Opc", "Inh", "Ex"]

# Expected chunk count per set.
# Inh chunk 100 had zero genes after filters -> 99 chunks only.
EXPECTED_CHUNKS_PER_SET = {
    "Ast": 100,
    "Mic": 100,
    "Oli": 100,
    "Opc": 100,
    "Inh": 99,
    "Ex":  100,   # per set; Ex has 3 sets
}

# How many sets per cell type (Ex split into 3 donor-disjoint sets, rest are set 1)
EXPECTED_SETS = {
    "Ast": {1}, "Mic": {1}, "Oli": {1}, "Opc": {1}, "Inh": {1},
    "Ex":  {1, 2, 3},
}

LOG2FC_THRESH = 0.25
Q_THRESH = 0.05

pattern = re.compile(
    r"poisson_DE_results_SEAAD_(?P<celltype>\w+)_set(?P<set>\d+)_chunk(?P<chunk>\d+)_of_(?P<nchunks>\d+)\.csv"
)

# ============================================================
# STEP 1 — DISCOVER + VERIFY CHUNKS
# ============================================================

files_by_cell_set = defaultdict(lambda: defaultdict(dict))

for fname in os.listdir(INPUT_DIR):
    m = pattern.match(fname)
    if not m:
        continue
    cell = m.group("celltype")
    if cell not in CHUNKED_CELLTYPES:
        continue
    set_id = int(m.group("set"))
    chunk  = int(m.group("chunk"))
    files_by_cell_set[cell][set_id][chunk] = os.path.join(INPUT_DIR, fname)

for cell in CHUNKED_CELLTYPES:
    if cell not in files_by_cell_set:
        raise RuntimeError(f"{cell}: no files found at all")
    found_sets = set(files_by_cell_set[cell].keys())
    if found_sets != EXPECTED_SETS[cell]:
        raise RuntimeError(f"{cell}: sets found {found_sets} != expected {EXPECTED_SETS[cell]}")
    expected = EXPECTED_CHUNKS_PER_SET[cell]
    for set_id, chunks in files_by_cell_set[cell].items():
        missing = sorted(set(range(1, expected + 1)) - set(chunks.keys()))
        if missing:
            raise RuntimeError(f"{cell} set {set_id}: missing chunks {missing} (expected {expected})")

print("✅ All required chunks present.")

# ============================================================
# STEP 2 — LOAD + CONCATENATE ALL CHUNKS (keep all chunk columns)
# ============================================================

dfs = []
for cell in CHUNKED_CELLTYPES:
    for set_id, chunk_map in files_by_cell_set[cell].items():
        for chunk_id in sorted(chunk_map):
            df = pd.read_csv(chunk_map[chunk_id])
            # Keep the raw per-chunk columns; recompute p_adj/log2FC/DEG below.
            df = df[["gene", "estimate_AD", "se_AD", "pval_AD", "n_cells"]].copy()
            df["celltype"] = cell
            df["set_id"]   = set_id
            df["chunk_id"] = chunk_id
            dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)
print(f"Concatenated rows across all chunks: {len(all_df):,}")

# ============================================================
# STEP 3 — RECOMPUTE GLOBAL FDR PER CELL TYPE
# ============================================================

def recompute_fdr(df):
    df = df.copy()
    mask = df["pval_AD"].notna()
    qvals = np.full(len(df), np.nan)
    qvals[mask] = multipletests(df.loc[mask, "pval_AD"].values, method="fdr_bh")[1]
    df["p_adj"]  = qvals
    df["log2FC"] = df["estimate_AD"] / np.log(2)
    df["DEG"]    = (df["p_adj"] < Q_THRESH) & (df["log2FC"].abs() > LOG2FC_THRESH)
    return df

# ============================================================
# STEP 4 — WRITE COMBINED OUTPUTS
# ============================================================

for cell in CHUNKED_CELLTYPES:
    sub = all_df[all_df["celltype"] == cell].copy()

    # Chunks are disjoint by construction; guard against any duplicate (set_id, gene)
    sub = sub.sort_values("pval_AD").drop_duplicates(["set_id", "gene"], keep="first")

    if cell == "Ex":
        # ---------- Ex: 3 donor-disjoint sets -> inverse-variance fixed-effect meta-analysis ----------
        sub = sub.dropna(subset=["gene", "estimate_AD", "se_AD"])
        sub = sub[sub["se_AD"] > 0].copy()

        def _meta_one(g):
            beta = g["estimate_AD"].to_numpy(dtype=float)
            se   = g["se_AD"].to_numpy(dtype=float)
            w    = 1.0 / (se ** 2)
            beta_hat = (w * beta).sum() / w.sum()
            se_hat   = 1.0 / np.sqrt(w.sum())
            z = beta_hat / se_hat
            # two-sided p from N(0,1) via erf:  Phi(z) = 0.5*(1+erf(z/sqrt(2)))
            p = 2.0 * (1.0 - (0.5 * (1.0 + math.erf(abs(z) / math.sqrt(2.0)))))
            return pd.Series({
                "estimate_AD": beta_hat,
                "se_AD":       se_hat,
                "pval_AD":     p,
                "n_cells":     g["n_cells"].sum(),
                "n_sets_used": g["set_id"].nunique(),
            })

        combined = (
            sub.groupby("gene", as_index=True)
               .apply(_meta_one, include_groups=False)
               .reset_index()                       # brings 'gene' back as a column
        )
        combined["celltype"] = "Ex"
        combined["set_id"]   = "META"
        combined["chunk_id"] = "META"

        combined = recompute_fdr(combined)
    else:
        combined = recompute_fdr(sub)

    out_path = os.path.join(OUTPUT_DIR, f"poisson_DE_results_SEAAD_{cell}_COMBINED.csv")
    combined.to_csv(out_path, index=False)
    print(f"{cell}: genes={combined.shape[0]} | DEGs={combined['DEG'].sum()} | written -> {out_path}")

print("\n✅ DONE — SEA-AD DEG combination complete.")


✅ All required chunks present.
Concatenated rows across all chunks: 65,356
Ast: genes=6559 | DEGs=803 | written -> /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_DEG_results/Processed/poisson_DE_results_SEAAD_Ast_COMBINED.csv
Mic: genes=4496 | DEGs=398 | written -> /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_DEG_results/Processed/poisson_DE_results_SEAAD_Mic_COMBINED.csv
Oli: genes=4669 | DEGs=392 | written -> /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_DEG_results/Processed/poisson_DE_results_SEAAD_Oli_COMBINED.csv
Opc: genes=6395 | DEGs=199 | written -> /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_DEG_results/Processed/poisson_DE_results_SEAAD_Opc_COMBINED.csv
Inh: genes=9008 | DEGs=566 | written -> /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_DEG_results/Processed/poisson_DE_results_SEAAD_Inh_COMBINED.csv
Ex: genes=11687 | DEGs=1360 | written -> /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_DEG_results/Processed/poisso